# P10 — Update block 3: T4 cache·SAE/TC 학습·검산

**준비·학습 전달물이며 P10 완료가 아닙니다.** LM seed 0/1/2 × SAE/TC × k=4/16 = 12 runs, 각각 5,000 updates. train 50k / validation 10k Update 위치를 사용합니다. 초기화 SET은 제외합니다. 별도 Update 전처리·RNG를 사용하고 상태 라벨은 loss에 넣지 않습니다.

1. T4 GPU 런타임을 선택합니다. 입력 ZIP을 `내 드라이브/boolean_interp_v1_4/`에 업로드합니다.
2. 아래 순서대로 실행합니다. 새 출력은 `P10_update_l3_r1`에 저장됩니다. P5 cache는 필요 없습니다.
3. 추출/학습 셀이 시간 제한으로 PAUSED이면 같은 셀을 다시 실행합니다. 런타임이 끊겼다면 맨 위부터 다시 실행합니다. 완전 저장 chunk/checkpoint부터 재개합니다.
4. 학습·수치 검산 후 마지막 셀에서 ZIP과 checksum을 내려받아 반환합니다. 오류나 자원 중단 때도 마지막 셀로 증빙을 보존합니다. Drive 원본 cache와 inputs는 삭제하지 마세요.

잔여 예산은 24.59 CU로 기록했습니다. 차감률은 Colab UI의 현재 값을 사용하며 고정 환산하지 않습니다. 추출·학습은 호출당 기본 2시간, chunk/250-update 경계에서 정지합니다. CPU 분석·최종 평가는 별도 후속 전달물입니다. 학습 전에 `evaluation_rules.json`에 라벨·전이·선택을 동결했습니다. Update 개입 실험·추가 층·length·m→m SAE는 이번 자원 범위에서 생략합니다.

In [ ]:
from google.colab import files, drive
from pathlib import Path
import hashlib, zipfile, subprocess, sys, json, time, shutil, os
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/boolean_interp_v1_4')
BUNDLE=BASE/'v1_4_p10_update_training_r1.zip'
assert BUNDLE.exists(), '입력 ZIP을 위 Drive 경로에 업로드하세요'
def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''): h.update(b)
    return h.hexdigest()
assert digest(BUNDLE)=='58be7323e59474e659cf350034900c6e3eaf7692fd884462c3c4cddbc8bdbdf9', '입력 ZIP checksum 불일치'
ROOT=Path('/content/MI_P10_update_r1');ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    for n in z.namelist(): assert (ROOT/n).resolve().is_relative_to(ROOT.resolve()), 'ZIP 경로 오류'
    z.extractall(ROOT)
OUTPUT=BASE/'P10_update_l3_r1';OUTPUT.mkdir(parents=True,exist_ok=True)
print('checksum 검증 완료; 출력:',OUTPUT)

In [ ]:
log=subprocess.run([sys.executable,'-m','pip','install','numpy==2.1.3','scipy==1.16.3','pytest==8.4.2','torch==2.11.0','--extra-index-url','https://download.pytorch.org/whl/cu128'],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
(OUTPUT/f'install_{time.time_ns()}.txt').write_text(log.stdout)
print(log.stdout);log.check_returncode()
os.environ['OMP_NUM_THREADS']='2';os.environ['OPENBLAS_NUM_THREADS']='2';os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
sys.path.insert(0,str(ROOT))
from interp_v1_4.p10 import verify, export
verify(ROOT)
def run(args):
    path=OUTPUT/'logs'/f'{time.time_ns()}.txt';path.parent.mkdir(exist_ok=True)
    with path.open('w') as log:
        proc=subprocess.Popen([sys.executable,'-u']+args,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        try:
            for line in proc.stdout: print(line,end='');log.write(line);log.flush()
            code=proc.wait()
        except BaseException:
            proc.terminate();proc.wait();raise
    if code:raise RuntimeError(f'실패 exit={code}; 로그 {path}; 마지막 셀로 증빙 반환')
run(['-m','pytest','tests_v1_4/test_p10.py','-q'])

## 현재 GPU와 비용 기록
`CU_PER_HOUR`는 Colab UI에 표시된 현재 차감률을 아는 경우만 입력하세요. 값을 입력하면 요청한 CU 예산과 호출 시간 중 작은 값으로 제한합니다. 실제 사용량은 Colab UI로 확인하고 잔여량을 갱신하세요. 아래 시간은 한 셀 호출의 상한이며 전체 분석 시간 예측이 아닙니다.

In [ ]:
REMAINING_CU=24.59  # 이후 세션에서는 UI의 실제 잔여량으로 갱신
CU_PER_HOUR=None  # UI의 현재 값, 예측값을 넣지 마세요
MAX_SECONDS=7200
if CU_PER_HOUR is not None:
    assert CU_PER_HOUR>0 and REMAINING_CU>0
    MAX_SECONDS=min(MAX_SECONDS,int(REMAINING_CU/CU_PER_HOUR*3600*0.8))
assert MAX_SECONDS>0
import torch
assert torch.cuda.is_available(), 'GPU 런타임을 선택하세요'
print('현재 GPU:',torch.cuda.get_device_name(),'; 호출 시간 제한(초):',MAX_SECONDS)
(OUTPUT/f'budget_{time.time_ns()}.json').write_text(json.dumps(dict(reported_remaining_cu=REMAINING_CU,live_rate=CU_PER_HOUR,max_seconds=MAX_SECONDS,gpu=torch.cuda.get_device_name()),indent=2))
SMOKE=OUTPUT/'smoke'/str(time.time_ns())
run(['-m','interp_v1_4.p10','smoke','--root',str(ROOT),'--output',str(SMOKE),'--device','cuda'])

## 1. Update train/validation cache 추출
각 LM의 동결 checkpoint를 사용합니다. 한 층의 h/u/m만 float32로 저장하며, metadata는 위치 선택·검증에만 사용합니다. 128개 sequence 단위로 Drive에 checksum과 함께 저장합니다. PAUSED이면 이 셀을 다시 실행하세요.

In [ ]:
run(['-m','interp_v1_4.p10','extract','--root',str(ROOT),'--output',str(OUTPUT),'--smoke',str(SMOKE/'smoke.json'),'--max-seconds',str(MAX_SECONDS)])

## 2. 입력 검사와 train 전용 scalar 통계
추출이 미완료이면 오류로 멈춥니다. 1번 셀부터 재개하세요. 9개 train 통계와 18개 train/val tensor를 검증합니다.

In [ ]:
run(['-m','interp_v1_4.p10','prepare','--root',str(ROOT),'--output',str(OUTPUT)])

## 3. SAE/TC 12 runs 학습
총 60,000 updates / 30.72M draws. 250 updates마다 전체 validation·checkpoint 저장, 100 update benchmark도 저장합니다. validation MSE 최소·동률은 이른 update를 선택합니다. PAUSED이면 이 셀을 다시 실행하세요. 전체 완료 전에는 다음 검산 셀이 성공하지 않습니다.

In [ ]:
run(['-m','interp_v1_4.p10','train','--root',str(ROOT),'--output',str(OUTPUT),'--smoke',str(SMOKE/'smoke.json'),'--max-seconds',str(MAX_SECONDS)])

## 4. 독립 수치 검산
252 checkpoint, 240 validation MSE, 선택·draw cursor·optimizer·RNG 및 9개 scalar 통계를 검증합니다. 학습 코드의 forward를 호출하지 않고 동결 연산 순서로 계산합니다. 실패 시 허용오차를 바꾸지 않고 증빙을 반환하세요.

In [ ]:
run([str(ROOT/'scripts/verify_v1_4_p10_training.py'),'--root',str(ROOT),'--output',str(OUTPUT),'--device','cuda'])

## 5. 완료 또는 중단 증빙 다운로드
전체 checkpoint·환경·로그·감사 결과를 반환합니다. 대형 cache/inputs 및 smoke tensor는 Drive에 유지하며 ZIP에서 제외합니다. 로컬 감사 후 별도 선택 checkpoint manifest와 의미·fidelity·전이 평가를 준비합니다. 새 평가에서 test로 재선택하지 않습니다.

In [ ]:
archive=Path(f'/content/v1_4_p10_training_return_{time.time_ns()}.zip')
export(OUTPUT,archive)
print('ZIP:',archive,'SHA256:',digest(archive))
exports=OUTPUT/'exports';exports.mkdir(exist_ok=True)
shutil.copyfile(archive,exports/archive.name)
shutil.copyfile(archive.with_suffix('.sha256.json'),exports/archive.with_suffix('.sha256.json').name)
files.download(str(archive));files.download(str(archive.with_suffix('.sha256.json')))